# Nhận diện khuôn mặt thời gian thực (FaceNet + MTCNN)

Hệ thống nhận diện khuôn mặt qua webcam sử dụng MTCNN để phát hiện và FaceNet để trích xuất đặc trưng (embedding).

## 1. Cơ sở lý thuyết
- **MTCNN:** Phát hiện vị trí khuôn mặt và 5 điểm mốc (mắt, mũi, miệng).
- **FaceNet:** Trích xuất vector 512 chiều đại diện cho đặc trưng khuôn mặt.
- **Cosine Similarity:** So sánh độ tương đồng giữa các vector (ngưỡng chuẩn: 0.7).

## 2. Cài đặt thư viện(chỉ chạy 1 lần rồi comment all)

In [1]:
# import subprocess
# import sys

# def install_package(package):
#     try:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
#         print(f"[OK] {package}")
#     except:
#         print(f"[FAIL] {package}")

# packages = ["opencv-python", "mtcnn", "keras-facenet", "scikit-learn", "Pillow", "numpy", "matplotlib", "tensorflow", "pandas"]
# for pkg in packages:
#     install_package(pkg)

## 3. Khai báo thư viện

In [2]:
import os
import sys
import time
import warnings
import csv
import cv2
import pickle
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from mtcnn import MTCNN
from keras_facenet import FaceNet

warnings.filterwarnings("ignore")

## 4. Khởi tạo mô hình

In [3]:
detector = MTCNN()# tải MTCNN
embedder = FaceNet()# tải FaceNet

## 5. Kiểm tra Webcam

In [4]:
def check_webcam(index=0):
    cap = cv2.VideoCapture(index)
    if not cap.isOpened(): return False
    ret, frame = cap.read()
    cap.release()
    return ret

CAMERA_INDEX = 0
if check_webcam(CAMERA_INDEX):
    print(f"Camera {CAMERA_INDEX} OK")
else:
    print("Không tìm thấy Camera")

Camera 0 OK


## 6. Các hàm xử lý khuôn mặt

In [5]:
FACE_SIZE = 160

# Phát hiện và cắt khuôn mặt
def extract_face(image, required_size=(FACE_SIZE, FACE_SIZE)):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(image_rgb)
    if not results: return None
    x, y, w, h = results[0]['box']
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = x1 + w, y1 + h
    face = image_rgb[y1:y2, x1:x2]
    return np.array(Image.fromarray(face).resize(required_size))

# Phát hiện tất cả khuôn mặt kèm tọa độ
def extract_all_faces(image, required_size=(FACE_SIZE, FACE_SIZE)):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(image_rgb)
    faces = []
    for res in results:
        if res['confidence'] < 0.9: continue
        x, y, w, h = res['box']
        x1, y1, x2, y2 = max(0, x), max(0, y), x + w, y + h
        face = np.array(Image.fromarray(image_rgb[y1:y2, x1:x2]).resize(required_size))
        faces.append((face, {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}))
    return faces

# Trích xuất embedding
def get_embedding(face_pixels):
    return embedder.embeddings(np.expand_dims(face_pixels, axis=0))[0]

# So sánh với database
def compare_face(embedding, database, threshold=0.7):
    if not database: return "Unknown", 0.0
    best_name, best_score = "Unknown", -1.0
    for name, ref_emb in database.items():
        score = cosine_similarity(embedding.reshape(1, -1), ref_emb.reshape(1, -1))[0][0]
        if score > best_score: best_score, best_name = score, name
    return (best_name, best_score) if best_score >= threshold else ("Unknown", best_score)

## 7. Quản lý Dataset và Database

In [6]:
DATASET_DIR = Path("../data")
DB_FILE = DATASET_DIR / "face_database.pkl"

# Xây dựng database từ ảnh trong thư mục
def build_database(dataset_dir):
    database = {}
    print("Đang quét ảnh và tạo Embeddings... (Có thể mất vài phút)")
    for person_dir in dataset_dir.iterdir():
        if not person_dir.is_dir() or person_dir.name.startswith('.'): continue
        embs = []
        for img_path in person_dir.glob("*.jpg"):
            img = cv2.imread(str(img_path))
            if img is not None:
                face = extract_face(img)
                if face is not None: embs.append(get_embedding(face))
        if embs: 
            database[person_dir.name] = np.mean(embs, axis=0)
            print(f"Đã xử lý: {person_dir.name}")
    return database

if DB_FILE.exists():
    print("Đang tải dữ liệu từ file lưu trữ...")
    with open(DB_FILE, 'rb') as f: database = pickle.load(f)
else:
    print("Không tìm thấy file lưu trữ. Bắt đầu tạo mới...")
    database = build_database(DATASET_DIR)
    with open(DB_FILE, 'wb') as f: pickle.dump(database, f)
    print("Đã lưu database vào file.")

print(f"Database: {list(database.keys())}")

Đang tải dữ liệu từ file lưu trữ...
Database: ['faceID', 'input', '0001_hy']


## 8. Giao diện và Điểm danh

In [7]:
def draw_result(frame, box, name, score):
    color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
    cv2.rectangle(frame, (box['x1'], box['y1']), (box['x2'], box['y2']), color, 2)
    cv2.putText(frame, f"{name} ({score:.2f})", (box['x1'], box['y1']-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

class AttendanceLogger:
    def __init__(self, path):
        self.path = Path(path)
        self.logged = set()
        if not self.path.exists():
            with open(self.path, 'w', newline='', encoding='utf-8') as f: 
                csv.writer(f).writerow(["Name", "Time", "Date"])

    def log(self, name):
        if name == "Unknown" or name in self.logged: return
        now = datetime.now()
        with open(self.path, 'a', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow([name, now.strftime("%H:%M:%S"), now.strftime("%Y-%m-%d")])
        self.logged.add(name)
        print(f"Điểm danh: {name}")

logger = AttendanceLogger(DATASET_DIR / "attendance_log.csv")

## 9. Đăng ký khuôn mặt mới

In [8]:
def register_new_person(frame, name, database, dataset_dir):
    face = extract_face(frame)
    if face is None: 
        print("Không tìm thấy khuôn mặt trong khung hình!")
        return database
    
    # 1. Tự động sinh ID tiếp theo (ví dụ 0001, 0002...)
    existing_ids = []
    for d in dataset_dir.iterdir():
        if d.is_dir() and '_' in d.name:
            try: existing_ids.append(int(d.name.split('_')[0]))
            except: pass
    next_id = max(existing_ids) + 1 if existing_ids else 1
    
    # 2. Tạo tên thư mục định dạng ID_Ten
    full_name = f"{next_id:04d}_{name}"
    person_path = dataset_dir / full_name
    person_path.mkdir(exist_ok=True)
    
    # 3. Lưu ảnh và cập nhật database
    cv2.imwrite(str(person_path / f"{int(time.time())}.jpg"), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
    emb = get_embedding(face)
    database[full_name] = (database[full_name] + emb) / 2 if full_name in database else emb
    
    # 4. Lưu lại database vào file pkl để lần sau không cần build lại
    with open(dataset_dir / "face_database.pkl", 'wb') as f:
        pickle.dump(database, f)
        
    print(f"Đã đăng ký thành công: {full_name}")
    return database

## 10. Nhận diện thời gian thực

In [9]:
def start_recognition(database, logger):
    cap = cv2.VideoCapture(CAMERA_INDEX)
    print("Q: Thoát | S: Đăng ký mới")
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        # Hiển thị hướng dẫn trên màn hình
        cv2.putText(frame, "Q: Thoat | S: Dang ky moi", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Tăng tốc bằng cách skip frame nếu cần
        faces = extract_all_faces(frame)
        for face_img, box in faces:
            emb = get_embedding(face_img)
            name, score = compare_face(emb, database)
            draw_result(frame, box, name, score)
            logger.log(name)
            
        cv2.imshow("Face Recognition", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        if key == ord('s'):
            name = input("Nhập tên: ")
            database = register_new_person(frame, name, database, DATASET_DIR)
            
    cap.release()
    cv2.destroyAllWindows()

# CHẠY CHƯƠNG TRÌNH
start_recognition(database, logger)

Q: Thoát | S: Đăng ký mới
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
Điểm danh: 0001_hy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
1/1 ━━━━━━━━━━━━━━━━

## 11. Xem kết quả điểm danh

In [10]:
import pandas as pd
if Path("../data/attendance_log.csv").exists():
    df = pd.read_csv("../data/attendance_log.csv")
    print(df.tail())
else:
    print("Chưa có dữ liệu điểm danh.")

       Name      Time        Date
2    kieumy  20:05:44  2026-04-21
3   0001_hy  20:37:17  2026-04-21
4   0001_hy  20:43:03  2026-04-21
5   0001_hy  20:49:40  2026-04-21
6  0004_mía  20:50:23  2026-04-21
